# Chapter 19 — Monte Carlo Simulation for Engineering
*Track 4: Engineers*

## 🎯 Learning Objectives
- Apply Monte Carlo to structural and tolerance analysis
- Propagate uncertainty through engineering calculations
- Estimate probabilities of failure for complex systems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Monte Carlo for Uncertainty Propagation

When $Y = g(X_1, X_2, \ldots, X_n)$ and the $X_i$ are random:

1. Sample $N$ realisations of each $X_i$
2. Compute $Y^{(j)} = g(X_1^{(j)}, \ldots, X_n^{(j)})$ for each $j$
3. Estimate $P(Y > \text{limit})$, $E[Y]$, $\text{Var}[Y]$

Advantages: handles **any** distribution, **any** functional form,
correlations, discrete + continuous mixing.

In [ ]:
# Structural example: beam deflection
# δ = P*L³ / (48*E*I) — all inputs are uncertain
N = 100_000
P = rng.normal(10_000, 1_000, N)   # Load [N]: mean=10kN, COV=10%
L = rng.normal(3.0, 0.05, N)       # Length [m]: mean=3m, COV=1.7%
E = rng.normal(2e11, 1e10, N)      # Elastic modulus [Pa]: steel
I = rng.normal(8.3e-6, 5e-7, N)    # Second moment [m⁴]

delta = P * L**3 / (48 * E * I)

limit = 0.010  # 10mm deflection limit
p_exceed = np.mean(delta > limit)

print(f"Mean deflection:     {delta.mean()*1000:.2f} mm")
print(f"Std deflection:      {delta.std()*1000:.2f} mm")
print(f"P(δ > {limit*1000:.0f}mm): {p_exceed:.4f}")

plt.figure(figsize=(9, 4))
plt.hist(delta*1000, bins=80, density=True, alpha=0.7, edgecolor="k", linewidth=0.2)
plt.axvline(limit*1000, color="red", lw=2, linestyle="--", label=f"Limit={limit*1000:.0f}mm")
plt.xlabel("Deflection (mm)"); plt.ylabel("Density")
plt.title(f"Monte Carlo Beam Deflection — P(exceed limit)={p_exceed:.4f}")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Tolerance Stackup

For linear stacking of $n$ dimensions:
$$Y = \sum_{i=1}^n X_i$$

If $X_i \sim N(\mu_i, \sigma_i^2)$ independently:
$$Y \sim N\left(\sum \mu_i, \sum \sigma_i^2\right)$$

**Worst-case analysis** assumes all at extreme:
$$\text{Total tolerance} = \sum_i t_i$$

**RSS (Root Sum of Squares):**
$$\text{Total tolerance}_{RSS} = \sqrt{\sum_i t_i^2}$$

In [ ]:
# Tolerance stackup: shaft in housing
n_parts = 6
means = np.array([50.0, 30.0, 20.0, 15.0, 10.0, 5.0])  # nominal dimensions
tolerances = np.array([0.05, 0.04, 0.03, 0.03, 0.02, 0.02])  # ±tolerance (3σ)
sigmas = tolerances / 3

# Monte Carlo stackup
N = 200_000
dims = rng.normal(means, sigmas, (N, n_parts))
total = dims.sum(axis=1)

worst_case_tol = tolerances.sum()
rss_tol = np.sqrt((tolerances**2).sum())

print(f"Nominal total:     {means.sum():.1f}")
print(f"Worst-case ±:      {worst_case_tol:.4f}")
print(f"RSS ±:             {rss_tol:.4f}")
print(f"MC std:            {total.std():.4f}")
print(f"MC ±3σ:            {3*total.std():.4f}")

clearance_limit = means.sum() + rss_tol * 1.5
p_exceed = np.mean(total > clearance_limit)
print(f"
P(total > clearance {clearance_limit:.2f}): {p_exceed:.6f}")

In [ ]:
# Sensitivity analysis: which input drives the most variance?
from scipy.stats import spearmanr
for i, name in enumerate(["P", "L", "E", "I"]):
    inputs = [P, L, E, I]
    rho, p_val = spearmanr(inputs[i], delta)
    print(f"Rank correlation of {name} with deflection: {rho:.4f}")

In [ ]:
# Reliability index β (First-Order Reliability Method approximation)
# For linear limit state g = R - S (resistance minus stress)
mu_R, sig_R = 150_000, 15_000   # Resistance
mu_S, sig_S = 100_000, 20_000   # Stress (load)

beta = (mu_R - mu_S) / np.sqrt(sig_R**2 + sig_S**2)
pf_approx = stats.norm.sf(beta)

# Monte Carlo verification
R_mc = rng.normal(mu_R, sig_R, 500_000)
S_mc = rng.normal(mu_S, sig_S, 500_000)
pf_mc = np.mean(R_mc < S_mc)

print(f"Reliability index β = {beta:.3f}")
print(f"FORM P(failure) = {pf_approx:.6f}")
print(f"MC   P(failure) = {pf_mc:.6f}")